In [1]:
# Program to show how LLM can call a tool based on the requirement
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, ToolMessage, AIMessage, HumanMessage
import requests
from langchain_core.tools import tool

In [2]:
@tool
def get_weather(latitude: float, longitude: float) -> dict:
    """function to return weather information based on latitude and longitude"""

    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    )
    data = response.json()
    return data["current_weather"]


In [3]:
llm = init_chat_model("gpt-5-nano", model_provider="openai")

system_message = SystemMessage(
    "You are a helpful weather assistant.If needed, find out latitude and logitude then provide them to tool. "
)
user_message = HumanMessage("What is the weather in Perth, Australia")
# user_message = HumanMessage("What is the capital of Australia")

messages = [system_message, user_message]

In [4]:
llm_with_tools = llm.bind_tools([get_weather])

In [5]:
while True:
    ai_msg = llm_with_tools.invoke(messages)
    # Add AI message to messages
    messages.append(ai_msg)

    # Terminate loop if no tool is to be called
    if len(ai_msg.tool_calls) == 0:
        break

    # call tool selected by LLM
    for tool_call in ai_msg.tool_calls:
        selected_tool = eval(tool_call["name"])
        tool_result = selected_tool.invoke(tool_call)
        messages.append(tool_result)

In [6]:
# print all messages
for message in messages:
    message.pretty_print()

================================ System Message ================================

You are a helpful weather assistant.If needed, find out latitude and logitude then provide them to tool. 
================================ Human Message =================================

What is the weather in Perth, Australia
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Tg6G5wQgzWOx4NAl6PIGxRoY)
 Call ID: call_Tg6G5wQgzWOx4NAl6PIGxRoY
  Args:
    latitude: -31.9523
    longitude: 115.8613
================================= Tool Message =================================
Name: get_weather

{"time": "2026-10-24T12:45", "interval": 900, "temperature": 13.5, "windspeed": 17.3, "winddirection": 176, "is_day": 0, "weathercode": 0}
================================== Ai Message ==================================

Current weather in Perth, Australia (lat -31.9523, lon 115.8613):
- Observation time: 2026-10-24 12:45
- Sky: Clear
- Temperature: 13.5